# Лабораторная работа 3. Структуры данных

**Опора:** лекция 6.

Сначала реализуются массив фиксированной ёмкости и односвязный список, затем
два стека поверх этих структур. После этого реализуются две хеш-функции и
хеш-таблица с цепочками. Генерация данных, тесты, измерения и графики находятся
в пакете `labkit` рядом с ноутбуком.

In [ ]:
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "labkit").is_dir():
        sys.path.insert(0, str(candidate))
        break

import labkit as lk

print("Режим:", lk.CONFIG.name, "| размеры структур:", lk.sizes())

## Задача 1. Массив фиксированной ёмкости

Структура хранит последовательность элементов во внутреннем хранилище, созданном
один раз на `capacity` элементов. После создания длина хранилища не меняется:
добавление и удаление меняют только счётчик занятых ячеек и сдвигают элементы.

Поддерживаются добавление в конец, вставка по индексу, удаление по индексу и
доступ по индексу. Выход за границы и переполнение — ошибка.

- **Требование:** методы, меняющие длину внутреннего списка (`append`, `insert`,
  `pop`, `remove`, `clear` у самого списка), не используются. Тест проверяет это
  отдельно.

In [ ]:
class FixedArray:
    """Массив фиксированной ёмкости.

    Хранилище создаётся один раз на capacity ячеек и больше не меняет длину.
    Число занятых ячеек хранится отдельно.
    """

    def __init__(self, capacity):
        self.capacity = capacity
        self.data = [None] * capacity
        self.size = 0

    def __len__(self):
        return self.size

    def get(self, index):
        """Значение по индексу; выход за границу — ошибка."""
        # TODO: реализовать
        raise NotImplementedError("FixedArray.get")

    def append(self, value):
        """Добавить в конец; переполнение — ошибка."""
        # TODO: реализовать
        raise NotImplementedError("FixedArray.append")

    def insert(self, index, value):
        """Вставить по индексу со сдвигом вправо."""
        # TODO: реализовать
        raise NotImplementedError("FixedArray.insert")

    def remove_at(self, index):
        """Удалить по индексу со сдвигом влево, вернуть удалённое значение."""
        # TODO: реализовать
        raise NotImplementedError("FixedArray.remove_at")

In [ ]:
lk.check_fixed_array(FixedArray)

## Задача 2. Односвязный список

Реализовать односвязный список. Поддерживаются вставка в начало, вставка
в конец, удаление первого элемента и доступ по индексу.

In [ ]:
class SinglyLinkedList:
    """Односвязный список."""

    def __init__(self):
        self.head = None
        self.tail = None
        self.size = 0

    def __len__(self):
        return self.size

    def append(self, value):
        """Добавить в конец."""
        raise NotImplementedError("SinglyLinkedList.append")

    def prepend(self, value):
        """Добавить в начало."""
        raise NotImplementedError("SinglyLinkedList.prepend")

    def get(self, index):
        """Значение по индексу; выход за границу — ошибка."""
        raise NotImplementedError("SinglyLinkedList.get")

    def remove_first(self):
        """Удалить первый элемент и вернуть его; пустой список — ошибка."""
        raise NotImplementedError("SinglyLinkedList.remove_first")

In [ ]:
lk.check_linked_list(SinglyLinkedList, "Односвязный список")

## Задача 3. Стек на подготовленных структурах

Реализуются **два стека**: один поверх собственного массива фиксированной
ёмкости, другой поверх собственного связного списка. Использовать нужно
структуры из задач 1 и 2, а не список Python.

Поддерживаются добавление элемента, извлечение последнего добавленного, просмотр
последнего добавленного и проверка на пустоту. Извлечение из пустого стека —
ошибка.

In [ ]:
class ArrayStack:
    """Стек поверх собственного массива фиксированной ёмкости."""

    def __init__(self, capacity):
        self.data = FixedArray(capacity)

    def __len__(self):
        return len(self.data)

    def is_empty(self):
        raise NotImplementedError("ArrayStack.is_empty")

    def push(self, value):
        raise NotImplementedError("ArrayStack.push")

    def pop(self):
        """Извлечь последний добавленный; пустой стек — ошибка."""
        raise NotImplementedError("ArrayStack.pop")

    def top(self):
        """Посмотреть последний добавленный; пустой стек — ошибка."""
        raise NotImplementedError("ArrayStack.top")


class LinkedStack:
    """Стек поверх собственного связного списка.

    Ёмкость связному списку не нужна, но аргумент принимается: так обе
    реализации создаются одинаково.
    """

    def __init__(self, capacity=None):
        # TODO: использовать реализованный выше связный список
        self.data = None

    def __len__(self):
        return len(self.data)

    def is_empty(self):
        raise NotImplementedError("LinkedStack.is_empty")

    def push(self, value):
        raise NotImplementedError("LinkedStack.push")

    def pop(self):
        raise NotImplementedError("LinkedStack.pop")

    def top(self):
        raise NotImplementedError("LinkedStack.top")

In [ ]:
lk.check_stack(ArrayStack, "Стек на массиве")
lk.check_stack(LinkedStack, "Стек на связном списке")

## Задача 4. Две хеш-функции для строк

Ключи в этой работе строковые. Реализовать **обе** функции: они различаются
только тем, учитывают ли порядок символов, и это различие — предмет
эксперимента.

**Сумма кодов** — намеренно плохая функция:

```text
h(s) = (сумма кодов символов s) mod m
```

**Полиномиальный хеш** — схема Горнера с основанием `X`, выданным в шаблоне:

```text
h = 0
для каждого символа c строки: h = (h * X + ord(c)) mod m
```

Обе функции принимают ключ и число слотов `m` и возвращают целое из
диапазона `0 … m-1`.

In [ ]:
X = 31   # основание полиномиального хеша


def hash_sum(key, m):
    """Сумма кодов символов по модулю m.

    Порядок символов эта функция не учитывает — в этом и состоит предмет
    сравнения с полиномиальным хешем.
    """
    # TODO: реализовать
    raise NotImplementedError("hash_sum")


def hash_poly(key, m):
    """Полиномиальный хеш по схеме Горнера с основанием X.

    h = 0, и для каждого символа строки: h = (h * X + ord(символ)) % m
    """
    # TODO: реализовать
    raise NotImplementedError("hash_poly")

In [ ]:
lk.check_hash_functions(hash_sum, hash_poly)

## Задача 5. Хеш-таблица с цепочками

Реализовать хеш-таблицу с разрешением коллизий методом цепочек. Хеш-функция
передаётся при создании, поэтому таблица работает с любой из двух функций
задачи 4:

```python
table = HashTableChaining(capacity, hash_poly)
```

Каждый слот хранит список всех пар, чьи ключи в него хешировались. Таблица
поддерживает добавление пары «ключ — значение» (повторный ключ обновляет
значение), поиск значения по ключу и проверку наличия ключа.
Метод `find` возвращает найденное значение, а если ключа нет — `None`.

In [ ]:
class HashTableChaining:
    """Хеш-таблица с цепочками; хеш-функция задаётся при создании."""

    def __init__(self, capacity, hash_function):
        self.capacity = capacity
        self.hash_function = hash_function
        self.buckets = [[] for _ in range(capacity)]
        self.size = 0

    def __len__(self):
        return self.size

    def insert(self, key, value):
        """Добавить пару; существующий ключ — обновить значение."""
        # TODO: реализовать
        raise NotImplementedError("HashTableChaining.insert")

    def find(self, key):
        """Значение по ключу; если ключа нет — None."""
        # TODO: реализовать
        raise NotImplementedError("HashTableChaining.find")

    def contains(self, key):
        return self.find(key) is not None

In [ ]:
lk.check_hash_table(HashTableChaining, hash_poly, "Цепочки")

## Набор реализованных структур

Дальше всё считается по тем реализациям, которые действительно написаны.

In [ ]:
LISTS = lk.implemented(
    {"односвязный список": SinglyLinkedList},
    probe=lambda cls: cls().append(1),
)
TABLES = lk.implemented(
    {"цепочки": HashTableChaining},
    probe=lambda cls: cls(8, hash_poly).insert("a", 1),
)
HASHES = lk.implemented(
    {"сумма кодов": hash_sum, "полиномиальный": hash_poly},
    probe=lambda function: function("a", 8),
)

print("Списки:", ", ".join(LISTS) or "нет")
print("Хеш-таблицы:", ", ".join(TABLES) or "нет")
print("Хеш-функции:", ", ".join(HASHES) or "нет")

## Эксперимент 1. Время операций

Четыре операции при разных размерах структуры. Линия показывает медианное
время одной операции, полоса — интервал от первого до третьего квартиля.

In [ ]:
operations = lk.operations_table(FixedArray, LISTS)
lk.plot_operations(operations);

## Эксперимент 2. Две хеш-функции в таблице с цепочками

Таблица с цепочками заполняется по очереди с каждой из двух хеш-функций при
коэффициентах заполнения `0.25, 0.5, 0.7, 0.9`. Число слотов фиксировано и
простое, меняется только число ключей. Ключи — случайные строки одинаковой
длины. Сравниваются доля ключей с коллизией и время успешного и неуспешного
поиска.

In [ ]:
if TABLES and HASHES:
    random_keys = lk.hash_grid_table(TABLES, HASHES, "случайные строки")
    lk.plot_hash_grid(random_keys);
else:
    print("Хеш-таблица или хеш-функция не реализована — эксперимент пропущен")

## Самостоятельные выводы

Сформулируйте выводы по полученным графикам.

1. Как меняются затраты каждой операции массива и односвязного списка
   с размером структуры? Объясните различия по коду.
2. Какие операции имеют примерно постоянное время, а какие растут вместе с `n`?
3. Почему `push`, `pop` и `top` могут выполняться за ожидаемое `O(1)` в обоих
   стеках? Какой конец массива и списка должен играть роль вершины стека?
4. Чем стек на массиве отличается от стека на связном списке по ограничению
   ёмкости, дополнительной памяти и размещению элементов в памяти? В каких
   условиях разумнее выбрать каждую реализацию?
5. Как выбор хеш-функции влияет на долю коллизий и время поиска случайных
   строк? Почему хеш с меньшим числом коллизий не обязательно быстрее при
   небольшой заполненности?
6. Как устроена открытая адресация с линейным пробированием
   `i_j = (h(key) + j) % capacity` и почему поиск отсутствующего ключа можно
   остановить на первом свободном слоте?
7. Как изменились бы графики успешного и неуспешного поиска, если вместо
   цепочек использовать линейное пробирование? Почему заполненность `0.9`
   особенно опасна из-за образования кластеров?